In [ ]:
import os, csv, json
from datetime import datetime, timedelta, timezone

import anthropic
import psycopg
from dotenv import find_dotenv, load_dotenv
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool

# override=True makes .env authoritative. Without it, load_dotenv() leaves any
# ANTHROPIC_API_KEY already present in the kernel's environment untouched, so a
# kernel started before .env was fixed keeps using the stale key forever.
ENV_PATH = find_dotenv(usecwd=True)
load_dotenv(ENV_PATH, override=True)
DATABASE_URL = os.environ["DATABASE_URL"]
CURRENT_CUSTOMER_ID = 102

ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "").strip().strip("\"'")
if not ANTHROPIC_API_KEY.startswith("sk-ant-") or len(ANTHROPIC_API_KEY) < 90:
    raise ValueError(
        f"ANTHROPIC_API_KEY is unusable: got {len(ANTHROPIC_API_KEY)} characters, "
        f"expected 100+ starting with 'sk-ant-'. Loaded from: {ENV_PATH or 'NO .env FOUND'}\n"
        "  - 0 characters  -> no ANTHROPIC_API_KEY line in .env, or .env was not found\n"
        "  - short value   -> the key is truncated; paste the FULL key onto ONE line, no quotes, no line breaks\n"
        "Get a key at https://console.anthropic.com/settings/keys"
    )

llm = ChatAnthropic(
    model="claude-opus-5-5",
    api_key=ANTHROPIC_API_KEY,
    max_tokens=16000,
)
print("Model configured. Current demo customer:", CURRENT_CUSTOMER_ID)
print("API key loaded: %d characters" % len(ANTHROPIC_API_KEY))

In [2]:
create_tables_sql = """
CREATE TABLE IF NOT EXISTS customers (
    customer_id BIGINT PRIMARY KEY,
    name TEXT NOT NULL,
    email TEXT NOT NULL
);
CREATE TABLE IF NOT EXISTS orders (
    order_id TEXT PRIMARY KEY,
    customer_id BIGINT NOT NULL REFERENCES customers(customer_id),
    restaurant TEXT NOT NULL,
    items TEXT NOT NULL,
    status TEXT NOT NULL,
    total_amount NUMERIC(10,2) NOT NULL,
    created_at TIMESTAMPTZ NOT NULL,
    promised_at TIMESTAMPTZ,
    delivered_at TIMESTAMPTZ
);
"""
with psycopg.connect(DATABASE_URL, connect_timeout=5) as conn:
    conn.execute(create_tables_sql)
    expected = {r[0] for r in conn.execute("SELECT column_name FROM information_schema.columns WHERE table_name='orders'").fetchall()}
    assert {"order_id", "customer_id", "status", "total_amount"}.issubset(expected), "Unexpected orders schema"


def read_csv(file_name):
    with open("data/" + file_name, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

customers = read_csv("customers.csv")
orders = read_csv("orders.csv")
now = datetime.now(timezone.utc)
with psycopg.connect(DATABASE_URL, connect_timeout=5) as conn:
    conn.execute("TRUNCATE orders, customers CASCADE")
    for row in customers:
        conn.execute("INSERT INTO customers(customer_id, name, email) VALUES (%s,%s,%s)",
                     (int(row["customer_id"]), row["name"], row["email"]))
    for row in orders:
        created = now - timedelta(hours=float(row["created_hours_ago"]))
        promised = created + timedelta(minutes=int(row["promised_minutes"]))
        delivered = None if not row["delivery_minutes"] else created + timedelta(minutes=int(row["delivery_minutes"]))
        conn.execute("""INSERT INTO orders(order_id, customer_id, restaurant, items, status, total_amount, created_at, promised_at, delivered_at)
                        VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)""",
                     (row["order_id"], int(row["customer_id"]), row["restaurant"], row["items"], row["status"],
                      row["total_amount"], created, promised, delivered))
print("Loaded", len(customers), "customers and", len(orders), "orders")

Loaded 3 customers and 6 orders


##### Functions for Tools

In [3]:
def get_connection():
    return psycopg.connect(DATABASE_URL)

ORDER_COLUMNS = "order_id, restaurant, items, status, total_amount, created_at, promised_at, delivered_at"

def order_from_row(row):
    minutes_late = None
    if row[6] and row[7]:
        minutes_late = round((row[7] - row[6]).total_seconds() / 60)
    return {
        "order_id": row[0], "restaurant": row[1], "items": row[2], "status": row[3],
        "total_amount": float(row[4]), "created_at": str(row[5]),
        "promised_at": str(row[6]) if row[6] else None,
        "delivered_at": str(row[7]) if row[7] else None,
        "minutes_late": minutes_late,
    }

def query_latest_order(customer_id):
    with get_connection() as conn:
        return conn.execute(
            f"SELECT {ORDER_COLUMNS} FROM orders WHERE customer_id=%s ORDER BY created_at DESC LIMIT 1",
            (customer_id,),
        ).fetchone()

def query_order(customer_id, order_id):
    with get_connection() as conn:
        return conn.execute(
            f"SELECT {ORDER_COLUMNS} FROM orders WHERE order_id=%s AND customer_id=%s",
            (order_id, customer_id),
        ).fetchone()

In [4]:
@tool
def get_latest_order() -> dict:
    """Get the latest order belonging to the current classroom customer."""
    row = query_latest_order(CURRENT_CUSTOMER_ID)
    if row is None:
        return {"found": False, "message": "No orders were found for this customer."}
    return {"found": True, "order": order_from_row(row)}

@tool
def get_order_details(order_id: str) -> dict:
    """Get details of an order ID belonging to the current classroom customer."""
    row = query_order(CURRENT_CUSTOMER_ID, order_id)
    if row is None:
        return {"found": False, "message": "No matching order was found."}
    return {"found": True, "order": order_from_row(row)}

In [5]:
tools = [get_latest_order, get_order_details]
llm_with_tools = llm.bind_tools(tools)
for t in tools:
    print(t.name, t.description, t.args)

get_latest_order Get the latest order belonging to the current classroom customer. {}
get_order_details Get details of an order ID belonging to the current classroom customer. {'order_id': {'title': 'Order Id', 'type': 'string'}}


##### Implement Tool Calling Agent Manually

In [6]:
SYSTEM_PROMPT = (
    "You are a Zomato support assistant. Use tools to verify order details. "
    "Never invent order information. If a lookup returns no result, explain that you could not find it."
)

API_ERROR_HINTS = {
    400: "the request was rejected -- if it mentions 'credit balance', add credits at https://console.anthropic.com/settings/billing",
    401: "the key sent to Anthropic is wrong, truncated, or revoked. If you just edited .env, RESTART THE KERNEL -- this kernel is still using the key it loaded at startup",
    403: "the key is recognised but not permitted for this request -- wrong workspace/model access, or a disabled key",
    404: "the model name is unknown to this account",
    429: "rate limited -- slow down and retry",
}


def run_basic_agent(question, max_rounds=5):
    messages = [SystemMessage(content=SYSTEM_PROMPT), HumanMessage(content=question)]
    tool_map = {tool_obj.name: tool_obj for tool_obj in tools}

    for _ in range(max_rounds):
        try:
            response = llm_with_tools.invoke(messages)
        except anthropic.APIStatusError as error:
            hint = API_ERROR_HINTS.get(error.status_code, "see https://docs.anthropic.com/en/api/errors")
            return f"ANTHROPIC API ERROR {error.status_code}: {hint}\nDetail: {error.message}"

        messages.append(response)

        if not response.tool_calls:
            return response.content

        for call in response.tool_calls:
            tool_obj = tool_map.get(call["name"])
            if tool_obj is None:
                result = {"error": "Unknown tool"}
            else:
                result = tool_obj.invoke(call["args"])

            messages.append(
                ToolMessage(
                    content=json.dumps(result, default=str),
                    tool_call_id=call["id"],
                )
            )

    return "I could not complete the request in this run. Please try again."


print(run_basic_agent("Where is my latest order?"))

ANTHROPIC API ERROR 400: the request was rejected -- if it mentions 'credit balance', add credits at https://console.anthropic.com/settings/billing
Detail: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011CgnNESpbk99cPXAz1hUAU'}
